# Laptop lab 3: Linear classification and preprocessing
*Build a pipeline that spots the premium listings*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, November 12, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/lab03.ipynb)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

This week you met linear classifiers: logistic regression and the linear SVM, with their penalty `C` (Lecture 5). And you met preprocessing: scaling, one-hot encoding, new features, and the pipeline that keeps all of it inside cross-validation (Lecture 6). Today you use both on the Airbnb listings of Labs 1 and 2.

**The story.** This time the question is not *how much* a listing costs, but *whether* it is a **premium** listing: more than €300 per night. You get 1,000 listings, in their own units this time, with text columns such as the neighborhood. As in Lab 2, 2,000 other listings stay locked away: the **test set**. At the end of the lab the tutor unlocks it, and every pipeline in the room is judged on listings it has never seen.

**Your mission.** Build one pipeline, preprocessing and model together, that classifies listings as premium or not, and estimate its accuracy on new listings. The warm-up tasks show you the pieces one by one; after that, you decide how to put them together.

## How this lab works

| You see | What you do |
|---|---|
| **Run** | Run the cell and look at the output. |
| **Your turn** | Fill in the dots `...`. The check below the cell shows an orange *Not yet* box until you do, and a green box when you are done. |
| **Toolbox** | Click to unfold a piece of code that you can copy and change. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss the answers together. |
| **Stop** | Wait here until the tutor says you can go on. |

A pipeline (Lecture 6) works like every model in scikit-learn:

```python
pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1))   # preprocessing, then the model
pipe.fit(X_train, y_train)          # fits the scaler, scales, fits the model
pipe.predict(X_new)                 # the class: 1 (premium) or 0 (not premium)
pipe.predict_proba(X_new)           # the probability of each class (logistic regression)
pipe.score(X_new, y_new)            # for classification: the accuracy, the share classified correctly
```

| Part | Minutes |
|---|---|
| 0. Setup | 5 |
| 1. Logistic regression on two features | 15 |
| 2. Does scaling matter? | 15 |
| 3. Text columns | 10 |
| 4. Your mission | 30 |
| 5. The test set is unlocked | 5 |
| 6. What happened? | together, 15 |
| Check your understanding | together, at the end |

## Task 0. Setup

Run the cell below. It loads the libraries, downloads the tools of this lab and the data from Inside Airbnb, and cleans them as in Lab 1. You should see a green box with a ✓.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier

# The tools of this lab are in the course lab repository;
# they also download the data from Inside Airbnb.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab03_tools.py", "lab03_tools.py")
from lab03_tools import *

X, y = load_training_data()    # your 1,000 listings: features X, and y = 1 for premium
setup_ok()

**Run.** Your data: one row per listing. `y` is the label: 1 if the listing costs more than €300 per night, 0 if not. `X` holds fourteen features, **in their own units**: unlike Lab 2, nothing has been scaled for you.

| Feature | Meaning |
|---|---|
| `accommodates`, `bedrooms`, `beds`, `bathrooms` | maximum number of guests; number of bedrooms, beds and bathrooms. The raw file has the bathrooms as text, such as `"1.5 baths"`; we turned them into numbers (Lecture 6, "Cleaning the raw survey"). |
| `minimum_nights`, `availability_365` | shortest stay allowed, nights still bookable in the coming year |
| `number_of_reviews`, `reviews_per_month`, `review_scores_rating` | number of reviews, in all and per month; average rating (1 to 5) |
| `host_listings`, `superhost` | listings of the host in Amsterdam; 1 if the host is a superhost |
| `km_to_center` | distance to Dam Square in km, as in Lab 2 |
| `room_type`, `neighbourhood` | **text**: the type of rental, and one of 22 neighborhoods |

The lists `NUMERIC` and `CATEGORICAL` hold the names of the number columns and of the two text columns.

In [ ]:
print(X.shape, "  share of premium listings:", round(y.mean(), 2))
X.head()

## Task 1. Logistic regression on two features

We start with two features, so that we can draw the model: the distance to the center, and the number of reviews per month, a sign of how often a listing is booked (Lecture 5, "A straight decision boundary").

**Your turn.** Fill in the `fit` line: the model should learn from the two columns of `two`, and the labels `y`.

In [ ]:
two = ["km_to_center", "reviews_per_month"]
logreg = LogisticRegression()
logreg.fit(..., y)                    # learn from the two features (which?) and the labels

print("coefficients:", logreg.coef_.round(2), "  intercept:", logreg.intercept_.round(2))

In [ ]:
# Check
check_logreg()

<details><summary><b>Hint</b> (click to unfold)</summary>

`logreg.fit(X[two], y)`: first the features, the two columns of `X`, then the labels, as for every model in scikit-learn.

</details>

**Your turn.** Now draw the model, as in Lecture 5 ("A straight decision boundary"). The recipe works for any classifier with two features: ask the model for the probability at every point of a fine grid, color the grid by that probability, draw the line where the probability is 0.5, and put the listings on top. Read the code, then fill in the three dots: the probability of the black line, and the two axis labels.

In [ ]:
# 1. A grid of 200 x 200 points that covers the plot
km = np.linspace(0, 8, 200)
reviews = np.linspace(0, 7, 200)
K, R = np.meshgrid(km, reviews)
grid = pd.DataFrame({"km_to_center": K.ravel(), "reviews_per_month": R.ravel()})

# 2. The probability of premium at every grid point, back in the shape of the grid
P = logreg.predict_proba(grid)[:, 1].reshape(K.shape)

fig, ax = plt.subplots(figsize=(7, 4.5))
# 3. Colors: the probability; and a black line where the probability is ...
filled = ax.contourf(K, R, P, levels=np.linspace(0, 1, 11), cmap="RdBu", alpha=0.6)
ax.contour(K, R, P, levels=[...], colors="black", linewidths=2)
fig.colorbar(filled, ax=ax, label="probability of premium")

# 4. The listings on top: red circles for not premium, blue triangles for premium
premium = y == 1
ax.scatter(X.loc[~premium, "km_to_center"], X.loc[~premium, "reviews_per_month"],
           marker="o", s=12, color="#C0002A", label="not premium")
ax.scatter(X.loc[premium, "km_to_center"], X.loc[premium, "reviews_per_month"],
           marker="^", s=12, color="#25567B", label="premium")

ax.set_xlabel(...)
ax.set_ylabel(...)
ax.set_xlim(0, 8)
ax.set_ylim(0, 7)
ax.legend(loc="upper right")
plt.show()

In [ ]:
# Check
check_boundary_plot()

<details><summary><b>Hint</b> (click to unfold)</summary>

The decision boundary is where the model hesitates: `levels=[0.5]`. For the labels, say what the axis shows, with its unit: for example `ax.set_xlabel("distance to the center (km)")` and `ax.set_ylabel("reviews per month")`.

</details>

**Run.** Two listings: one 1 km from the center with half a review per month, and one 4 km from the center with four reviews per month.

In [ ]:
new = pd.DataFrame({"km_to_center": [1.0, 4.0], "reviews_per_month": [0.5, 4.0]})
print("predict_proba:")
print(logreg.predict_proba(new).round(2))      # columns: P(not premium), P(premium)
print("predict:", logreg.predict(new))

**Think.** (a) Read the two coefficients: what does their sign tell you? Why could often-reviewed listings be cheaper? (b) For the first listing, `predict_proba` gives two numbers. What are they, and why does `predict` say 1? (c) The boundary is a straight line. Why can a logistic regression never draw a curved boundary in these two features? (Lecture 5, "A straight decision boundary".)

*Your answer:*

## Task 2. Does scaling matter?

Now all twelve number columns. Look at their ranges: `availability_365` runs from 0 to 365, `superhost` from 0 to 1.

In [ ]:
X[NUMERIC].describe().loc[["min", "max"]].round(1)

**Run.** $k$-NN with $k = 15$ on the twelve numbers, unscaled, with 5-fold cross-validation (Lecture 3). The *accuracy* is the share of listings classified correctly.

In [ ]:
knn_raw = KNeighborsClassifier(n_neighbors=15)
score_raw = cross_val_score(knn_raw, X[NUMERIC], y, cv=5).mean()
print("15-NN, unscaled: mean validation accuracy", round(score_raw, 3))

**Your turn.** Put a `StandardScaler()` in front of the same model with `make_pipeline` (Lecture 6, "Pipelines"), and compute its mean validation accuracy in the same way. The pipeline refits the scaler inside every fold, on the training part only.

In [ ]:
knn_scaled = make_pipeline(..., KNeighborsClassifier(n_neighbors=15))
score_scaled = ...                    # mean 5-fold cross-validation accuracy on X[NUMERIC], as above
print("15-NN, scaled:   mean validation accuracy", round(score_scaled, 3))

In [ ]:
# Check
check_scaled_knn()

<details><summary><b>Hint</b> (click to unfold)</summary>

`make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))`, and for the score the line of the cell above with `knn_scaled` instead of `knn_raw`.

</details>

**Run.** The same comparison for logistic regression.

In [ ]:
for name, model in [("unscaled", LogisticRegression(max_iter=1000)),
                    ("scaled  ", make_pipeline(StandardScaler(), LogisticRegression()))]:
    print("logistic regression,", name, round(cross_val_score(model, X[NUMERIC], y, cv=5).mean(), 3))

**Think.** (a) Why does scaling change $k$-NN so much? Which features decide who the "nearest" listings are without it? (b) What happens to logistic regression? Why can scaling still matter for it? (Lecture 6, "Does it matter?")

*Your answer:*

## Task 3. Text columns

`room_type` and `neighbourhood` are text. A model needs numbers, but numbering the 22 neighborhoods 1 to 22 would invent an order and distances that do not exist. One-hot encoding gives every category its own 0/1 column (Lecture 6, "The program column").

**Your turn.** Complete the column transformer: the scaler for the numbers, the one-hot encoder for the text columns. Then run the pipeline with logistic regression on all of `X`.

In [ ]:
prep = make_column_transformer(
    (StandardScaler(), NUMERIC),
    (OneHotEncoder(handle_unknown="ignore"), ...))      # which columns?
pipe_cat = make_pipeline(prep, LogisticRegression())
score_cat = cross_val_score(pipe_cat, X, y, cv=5).mean()
print("numbers + text columns: mean validation accuracy", round(score_cat, 3))

In [ ]:
# Check
check_categories()

<details><summary><b>Hint</b> (click to unfold)</summary>

The list of the two text columns is called `CATEGORICAL`.

</details>

**Think.** Compare `score_cat` with the scaled logistic regression of Task 2, on the numbers only. Did the two text columns help? Why could that be, for the room type in particular? (Hint: which numbers already say a lot about the type of rental?)

*Your answer:*

## Task 4. Your mission

Now it is your turn. Build **one pipeline**, preprocessing and model together, and estimate its accuracy on the locked listings.

- **The models:** `LogisticRegression(C=...)`, `LinearSVC(C=...)` (Lecture 5) or `KNeighborsClassifier(n_neighbors=...)` (Lecture 3).
- **The preprocessing:** scaling, one-hot encoding, a log of the long-tailed columns, products or squares of features (Lecture 6). Which columns you use is up to you.
- **The rule:** you may do anything with `X` and `y`. The test set stays locked. Your pipeline must work on the whole table `X`.
- **How you choose is yours:** one validation split, cross-validation, a grid search. Unfold the toolbox below for code that you can copy and change.

Write your experiments in the cells below; add cells if you need more. Keep a note of what you tried and what you found.

<details><summary><b>Toolbox</b> (click to unfold)</summary>

**A. Choose `C` with a grid search** (Lecture 5, "Choosing C"). In a pipeline, a parameter is named after its step: the step name, two underscores, the parameter name (Lecture 6, "Pipelines and grid search").

```python
pipe = make_pipeline(prep, LogisticRegression(max_iter=1000))
grid = GridSearchCV(pipe, {"logisticregression__C": [0.001, 0.01, 0.1, 1, 10, 100]}, cv=5)
grid.fit(X, y)
print(grid.best_params_, round(grid.best_score_, 3))
```

**B. A linear SVM** (Lecture 5, "Linear support vector machine"): the same, with `LinearSVC()` and `"linearsvc__C"`.

**C. $k$-NN** in a pipeline, with $k$ chosen by a grid search: `"kneighborsclassifier__n_neighbors": range(1, 61)`.

**D. A long tail?** Look at the histograms: `X[NUMERIC].hist(figsize=(12, 8), bins=30)`. For counts with a long tail, try the logarithm (Lecture 6, "A long tail"). `np.log1p` computes $\log(1 + x)$, which also works for zeros. A column transformer can treat three groups of columns differently:

```python
long_tail = ["minimum_nights", "number_of_reviews", "reviews_per_month", "host_listings"]
other = [c for c in NUMERIC if c not in long_tail]
prep_log = make_column_transformer(
    (make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), long_tail),
    (StandardScaler(), other),
    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
```

**E. Products and squares** (Lecture 6, "Interactions" and "Two fixes"): `make_pipeline(StandardScaler(), PolynomialFeatures(degree=2))` in place of `StandardScaler()` adds every square and every product of two columns. Many new features: let `C` keep them in check.

**F. Fewer features:** `LogisticRegression(penalty="l1", solver="liblinear", C=...)` sets many coefficients exactly to 0 (Lecture 5, "The L1 penalty").

**G. Only some columns?** Leave a column out of the column transformer: by default, the columns you do not name are dropped.

</details>

In [ ]:
# your experiments

In [ ]:
# more experiments

**Your turn.** Your submission. Fill in your pipeline, a few words on how you chose it, and the accuracy you expect on the locked listings. The last line fits your pipeline once more on **all** 1,000 training listings.

In [ ]:
my_pipeline = ...      # your final choice, for example make_pipeline(prep, LogisticRegression(C=1))
my_strategy = "..."    # how you chose it, in your own words
my_estimate = ...      # the accuracy you expect on new listings, for example 0.75

my_pipeline.fit(X, y)  # learn from all 1,000 training listings

In [ ]:
# Check
check_submission()

**Think.** Before the test set is unlocked: will your pipeline score higher or lower on it than your estimate? Why?

*Your answer:*

> **Stop.** Wait here until the tutor unlocks the test set. Meanwhile, compare your pipeline with your neighbor's: the same model? The same preprocessing?

## Task 5. The test set is unlocked

**Your turn.** When the tutor shows the password on the screen, type it in quotes and run the cell. Then run the next cell: it scores your pipeline on the 2,000 listings that no model has seen.

In [ ]:
X_test, y_test = unlock_test_set("...")    # the password on the screen

In [ ]:
test_accuracy = report(my_pipeline, my_estimate, X_test, y_test)

Tell the tutor your pipeline, how you chose it, your estimate and your test accuracy. We collect them on the screen.

> **As in Lab 2:** the test set has now been used. Changing your pipeline to do better on it would make its score an optimistic estimate.

## Task 6. What happened?

**Run.** The pieces of today, one by one: for each pipeline, the mean 5-fold cross-validation accuracy on your 1,000 listings, and the accuracy on the test set. The first row always predicts the most common class.

In [ ]:
check_unlocked(X_test)      # stops here while the test set is still locked
from sklearn.dummy import DummyClassifier
long_tail = ["minimum_nights", "number_of_reviews", "reviews_per_month", "host_listings"]
other = [c for c in NUMERIC if c not in long_tail]
numbers = make_column_transformer((StandardScaler(), NUMERIC))
numbers_raw = make_column_transformer(("passthrough", NUMERIC))
with_text = make_column_transformer((StandardScaler(), NUMERIC),
                                    (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
with_log = make_column_transformer((make_pipeline(FunctionTransformer(np.log1p), StandardScaler()), long_tail),
                                   (StandardScaler(), other),
                                   (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL))
candidates = {
    "always the most common class": DummyClassifier(),
    "15-NN, numbers, unscaled": make_pipeline(numbers_raw, KNeighborsClassifier(15)),
    "15-NN, numbers, scaled": make_pipeline(numbers, KNeighborsClassifier(15)),
    "logistic regression, numbers, scaled": make_pipeline(numbers, LogisticRegression()),
    "  + text columns": make_pipeline(with_text, LogisticRegression()),
    "  + log of the long tails": make_pipeline(with_log, LogisticRegression()),
    "  same, C = 0.00001": make_pipeline(with_log, LogisticRegression(C=0.00001)),
    "linear SVM, same features": make_pipeline(with_log, LinearSVC()),
}
rows = []
for name, m in candidates.items():
    rows.append({"pipeline": name,
                 "5-fold CV": cross_val_score(m, X, y, cv=5).mean(),
                 "test set": m.fit(X, y).score(X_test, y_test)})
pd.DataFrame(rows).set_index("pipeline").round(3)

**Think.** We discuss these together.

(a) How much did scaling give $k$-NN? Would you ever leave it out of a pipeline with $k$-NN?<br>
(b) Did the text columns help, by cross-validation? On the test set? Which of the two do you trust to decide, and why?<br>
(c) What does logistic regression with `C = 0.00001` do? Compare it with the first row.<br>
(d) Logistic regression or linear SVM: does the choice matter much here? When would you need logistic regression? (Lecture 5, "Logistic regression or linear SVM?")<br>
(e) We dropped every listing with a missing value, about one in six. Who might those listings be, and could that matter? (Lecture 7 is about this.)

*Your answer:*

## Check your understanding

Answer each question for yourself first, then unfold the answer. We go through them together at the end of the lab.

**1.** For a listing, `predict_proba` returns `[0.35, 0.65]`. What does `predict` return?

A. 0 (not premium) &nbsp; B. 1 (premium) &nbsp; C. 0.65 &nbsp; D. It depends on `C`

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** The second column is the probability of class 1, premium. It is above 0.5, so `predict` says premium (Lecture 5, "Logistic regression in scikit-learn").

</details>

**2.** A colleague fits a `StandardScaler` on all 1,000 listings, and then runs 5-fold cross-validation on the scaled data. What is wrong?

A. Nothing &nbsp; B. The validation folds helped to fit the scaler: a leak &nbsp; C. The scaler should be fitted on the test set &nbsp; D. Logistic regression needs no scaling

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** The validation fold shaped the mean and the standard deviation before it was used to validate. Put the scaler in a pipeline, and it is refitted inside every fold, on the training part only (Lecture 6, "Where does the scaler go in cross-validation?").

</details>

**3.** `neighbourhood` has 22 values. How many columns does `OneHotEncoder` make of it?

A. 1 &nbsp; B. 2 &nbsp; C. 21 &nbsp; D. 22

<details><summary><b>Answer</b> (click to unfold)</summary>

**D.** One 0/1 column per neighborhood. (Statisticians drop one column; with a penalized model we keep all of them, Lecture 6, "The program column".)

</details>

**4.** Unscaled, which feature has the largest influence on the distances of $k$-NN in this lab?

A. `superhost` &nbsp; B. `bathrooms` &nbsp; C. `availability_365` &nbsp; D. `km_to_center`

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** It runs from 0 to 365, so its differences are by far the largest numbers (Lecture 3, "Distances depend on the units"; Lecture 6, "Two debts from earlier lectures").

</details>

**5.** A hosting platform wants to show each host *the probability* that their listing counts as premium. Which model?

A. `LinearSVC` &nbsp; B. `LogisticRegression` &nbsp; C. $k$-NN with $k = 1$ &nbsp; D. Any of them

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** Logistic regression turns its score into a probability; a linear SVM gives only a class (Lecture 5, "Logistic regression or linear SVM?"). 1-NN gives probabilities of only 0 or 1.

</details>

## If you finish early

**A. Which features does the L1 penalty keep?** Fit `make_pipeline(prep, LogisticRegression(penalty="l1", solver="liblinear", C=0.05))` on `X, y`. The names of the columns after the preprocessing are in `pipe[0].get_feature_names_out()`, the coefficients in `pipe[-1].coef_[0]`. Which features survive? Try a few values of `C`.

In [ ]:
...

**B. Three classes.** `load_price_classes()` gives the same listings in three classes: budget (up to €200), middle (up to €350) and luxury. Fit `make_pipeline(prep, LogisticRegression())` on `X` and these labels, and look at `predict_proba` for the first five listings. How many columns, and what do they sum to? (Lecture 5, "Multiclass in scikit-learn".)

In [ ]:
...

## Where this is going

> Put every step that learns from data inside the pipeline: then cross-validation judges the preprocessing and the model together, honestly.

Today the preprocessing mattered more than the choice between linear models: scaling for $k$-NN, the logarithm for the long tails, and a penalty strong enough, but not too strong.

**Coming up.**

- Lecture 7 (Monday, November 16): missing values and feature selection. Today we dropped the listings with gaps; next week you fill them, inside the pipeline.
- Lecture 8 (Tuesday, November 17): decision trees and random forests, models that need no scaling at all.

**Reading.** Müller and Guido (2016), Chapter 2: "Linear models for classification"; Chapter 3: "Preprocessing and Scaling"; Chapter 4: "Categorical Variables" and "Univariate Nonlinear Transformations"; Chapter 6: "Building Pipelines".

---
*This lab practices Lectures 5 and 6 of Ş. İlker Birbil. Data: [Inside Airbnb](https://insideairbnb.com), Amsterdam, collected on September 16, 2026, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); downloaded directly from their website and cleaned as in Lab 1.*